In [ ]:
!pip install plotly

In [ ]:
!pip install nbformat

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
nutritionData = pd.read_csv("nutrition_data.csv")

In [4]:
# rename columns with units

nutritionData = nutritionData.rename(columns = {
    "Protein": "Protein (g)",
    "Sodium": "Sodium (mg)",
    "Saturated Fat": "Saturated Fat (g)",
    "Dietary Fiber": "Dietary Fiber (g)",
    "Iron": "Iron (mg)",
    "Cholesterol": "Cholesterol (mg)"
})

In [5]:
# not enough vitamin B12 data

nutritionData = nutritionData.drop(columns = ["Vitamin B12"])

In [6]:
# get rid of units in the data cells themselves

for col in ["Protein (g)", "Sodium (mg)", "Saturated Fat (g)", "Dietary Fiber (g)", "Iron (mg)", "Cholesterol (mg)"]:
    nutritionData[col] = nutritionData[col].str.replace(r"[^\d.]", "", regex=True)


In [7]:
# switch everything to being numbers

for col in ["Calories", "Protein (g)", "Sodium (mg)", "Saturated Fat (g)", "Dietary Fiber (g)", "Iron (mg)", "Cholesterol (mg)"]:
    nutritionData[col] = pd.to_numeric(nutritionData[col])

In [8]:
# normalize every product to 200 calories

# get all the columns with numbers
numeric_cols = nutritionData.select_dtypes(include="number").columns

# create a multiplier based on calories
multiplier = 200 / nutritionData["Calories"]

# multiply across rows (axis = 0)
nutritionData[numeric_cols] = nutritionData[numeric_cols].multiply(multiplier, axis=0)


We'll find Pareto-optimal products for different pairs of variables.

In [9]:
# write a function to get the pareto optimal products for the nutrients in nutrients

# nutrients is a list of nutrients: ["Protein (g)", "Sodium (mg)"]
# wants is a list of booleans that corresponds to whether we want more or less of each: [True, False]
def getParetoOptimal(nutrients, wants):

    n = len(nutritionData)
    k = len(nutrients)
    nutritionalContent = np.array([None] * k)
    comparisonNutritionalContent = np.array([None] * k)

    # list of indices for which the product is pareto optimal
    paretoOptimal = []

    for idx in range(n):

        is_pareto = True

        # fill up the nutritionalContent array
        # for example, nutritionalContent = [17.14,457.14]
        for nutIdx in range(k):
            nutritionalContent[nutIdx] = nutritionData.iloc[idx][nutrients[nutIdx]]
        
        # now, having filled up nutritionalContent, we compare to the nutritional content of all the others

        for comparisonIdx in range(n):
            
            if comparisonIdx == idx:
                continue

            for nutIdx in range(k):
                comparisonNutritionalContent[nutIdx] = nutritionData.iloc[comparisonIdx][nutrients[nutIdx]]
            
            # this gives us comparisonNutritionalContent = [23.52,423.52], for example


            at_least_as_good = np.where(wants, comparisonNutritionalContent >= nutritionalContent, comparisonNutritionalContent <= nutritionalContent)
            strictly_better = np.where(wants, comparisonNutritionalContent > nutritionalContent, comparisonNutritionalContent < nutritionalContent)
            
            if at_least_as_good.all() and strictly_better.any():
                is_pareto = False
                break
        
        if is_pareto:
            paretoOptimal.append(idx)

    return nutritionData.iloc[paretoOptimal]

In [ ]:
def plotPareto(nutrients, wants, title):
    pareto_df = getParetoOptimal(nutrients, wants)
    pareto_indices = pareto_df.index

    # copy the dataframe so we can add a column to indicate pareto optimality
    plot_df = nutritionData.copy()
    plot_df["Pareto"] = plot_df.index.isin(pareto_indices)
    plot_df["Pareto"] = plot_df["Pareto"].map({True: "Pareto", False: "Not Pareto"})

    fig = px.scatter(plot_df, x=nutrients[0], y=nutrients[1],
                 hover_name="Product", color="Brand",
                 symbol="Pareto",
                 symbol_map={"Pareto": "star", "Not Pareto": "circle"},
                 hover_data={nutrients[0]: ":.1f", nutrients[1]: ":.1f"},
                 custom_data=["URL"],
                 title=title)

    for trace in fig.data:
        trace.hovertemplate = trace.hovertemplate + "<br><i>Click to open</i>"

    return fig



In [11]:
poProSod = plotPareto(["Protein (g)", "Sodium (mg)"], [True, False], "Alternative Proteins Comparison: Protein and Sodium (per 200 Calories)")
poProSat = plotPareto(["Protein (g)", "Saturated Fat (g)"], [True, False], "Alternative Proteins Comparison: Protein and Saturated Fat (per 200 Calories)")
poProFib = plotPareto(["Protein (g)", "Dietary Fiber (g)"], [True, True], "Alternative Proteins Comparison: Protein and Dietary Fiber (per 200 Calories)")
poSodSat = plotPareto(["Sodium (mg)", "Saturated Fat (g)"], [False, False], "Alternative Proteins Comparison: Sodium and Saturated Fat (per 200 Calories)")
poSodFib = plotPareto(["Sodium (mg)", "Dietary Fiber (g)"], [False, True], "Alternative Proteins Comparison: Sodium and Dietary Fiber (per 200 Calories)")
poSatFib = plotPareto(["Saturated Fat (g)", "Dietary Fiber (g)"], [False, True], "Alternative Proteins Comparison: Saturated Fat and Dietary Fiber (per 200 Calories)")

In [12]:
poProSod.write_html("protein_sodium.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])
poProSat.write_html("protein_satfat.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])
poProFib.write_html("protein_fiber.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])
poSodSat.write_html("sodium_satfat.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])
poSodFib.write_html("sodium_fiber.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])
poSatFib.write_html("satfat_fiber.html", post_script=["""
    var plot = document.getElementsByClassName('plotly-graph-div')[0];
    plot.on('plotly_click', function(data){
        var url = data.points[0].customdata[0];
        window.open(url, '_blank');
    });
"""])